# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.text_processing_functions import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load data (model)

res_savename = "labelled_reports_DREF_target_meta-llama_llama-4-scout-17b-16e-instruct_v211025"
#"labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv"
#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"
response_df = pd.read_csv(DATA_OUT_LLMS /(res_savename + ".csv"))
#load data (labelled)
#res_savename = "labelled_reports_impacts_all_v080925.csv"
#response_df = pd.read_csv(DATA_LABELLED / res_savename)
savename = "nounit_quali_post_processed_" + res_savename


In [3]:
#get rid of nans
response_df_proc = cp.deepcopy(response_df)

response_df_proc = response_df_proc.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df_proc.columns else response_df_proc

In [4]:
#process impactValue
response_df_proc = response_df_proc.apply(parse_impact_value_precision, axis=1)

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:170: RuntimeWarning: All-NaN slice encountered
  min_value = np.nanmin(all_values)
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:171: RuntimeWarning: All-NaN slice encountered
  max_value = np.nanmax(all_values)


In [6]:
response_df_proc[response_df_proc["impactSubtype"] == "Access to Food"]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,valid_errors_dates,hazards,hazardsAnnotation,valid_errors_haz,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
18,Access to Food,NaN,NaN,NaN,NaN,NaN,[The stagnant water in low-lying areas and the...,0,[Pakistan],"[Sindh, Balochistan, Khyber Pakhtunkhwa]",...,0,[Flood],[Pakistan endured an exceptionally intense mon...,0,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
39,Access to Food,NaN,NaN,NaN,NaN,NaN,[The situation directly affected displaced ind...,0,[Sudan],"[Tokar, Port Sudan, Aqeeq, Red Sea State, Whit...",...,0,"[Flood, Epidemic, Conflict]","[On 25 August 2024, heavy rains led to widespr...",0,MDRSD034,Sudan,2025-03-17,https://go-api.ifrc.org/api/downloadfile/90654...,Flood,['DREF Operational Update Sudan Floods 2024 Di...
60,Access to Food,NaN,NaN,NaN,NaN,NaN,"[food shortages are worsening, many families i...",0,[Georgia],"[Guria, Buknari, Kvenobani, Basileti, Jumati, ...",...,0,"[Extreme cold temperature, Other storm]","[Since 21 February 2025, Western Georgia has b...",0,MDRGE019,Georgia,2025-03-16,https://go-api.ifrc.org/api/downloadfile/90652...,Extreme Winter Condition,['DREF Operation Georgia: Heavy Snowfall 2025 ...
80,Access to Food,NaN,NaN,NaN,NaN,NaN,[high competition for limited income opportuni...,0,[Mozambique],"[Tete, Gaza, Manica, Inhambane]",...,0,"[Drought, Tropical storm]",[Mozambique is currently experiencing severe e...,0,MDRMZ024,Mozambique,2024-10-02,https://adore.ifrc.org/Download.aspx?FileId=83...,Drought,['1 OPERATION UPDATE Mozambique | Drought Emer...
97,Access to Food,NaN,NaN,NaN,NaN,NaN,[Flooding has devastated agricultural lands an...,0,[Algeria],"[Béchar, Elbayadh, Beni Abbes, Tamanrasset, Ti...",...,0,"[Flood, Tropical storm]","[On September 8, 2024, a severe tropical distu...",0,MDRDZ011,Algeria,2024-09-22,https://adore.ifrc.org/Download.aspx?FileId=83...,Famine / Food Insecurity,['DREF Operation Algeria Flood 2024 Bechar App...
118,Access to Food,NaN,NaN,NaN,NaN,NaN,[The lack of communication with the community ...,0,[Cameroon],"[Far North region, Logone et Chari division, M...",...,0,"[Flood, Convective storm]",[The Far North region has been experiencing fl...,0,MDRCM039,Cameroon,2024-09-13,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operation Cameroon_Far North Floods Eva...
132,Access to Food,NaN,NaN,NaN,NaN,NaN,[The main affected families are farmers and fi...,0,[Benin],"[Lalo, Couffo, Adoukandji, Ahodjinnako, Ahomad...",...,0,[Flood],[Intense rainfall observed in the departments ...,0,MDRBJ019,Benin,2024-09-07,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operation Benin_Flood in Lalo Field vis...
148,Access to Food,NaN,NaN,NaN,NaN,NaN,[Over 90% of crop-farming households and 76% o...,0,[Nigeria],"[Bauchi, Kebbi, Sokoto, Zamfara]",...,0,[Flood],"[From August 8 to August 13, 2024, continuous ...",0,MDRNG041,Nigeria,2024-09-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operation Nigeria Floods DREF 2024 Floo...
180,Access to Food,NaN,NaN,NaN,NaN,NaN,[The food security situation was alarming espe...,0,[Rwanda],"[Western Province, Northern Province, Southern...",...,0,"[Flood, Mass movement]","[From 1 to 6 May, Rwanda experienced continuou...",0,MDRRW022,Rwanda,2024-09-05,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Final Report Rwanda - Floods and Landsl...
224,Access to Food,NaN,NaN,NaN,NaN,NaN,[The disruption of local markets has made it d...,0,[Uganda],"[Ntoroko District, Bweramule Sub-County, Butun...",...,0,"[Flood, Convective storm, Mass movement]","[In April 2024, the Eastern Uganda-Elgon regio...",0,MDRUG050,Uganda,2024-08-30,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...


In [5]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation"]
list_cols = [key for key in list_cols if key in response_df_proc.columns]
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [25]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(list_country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(list_country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [26]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [27]:
#reclassify impacType
#response_df_proc["impactSubtype_orig"] = response_df_proc["impactSubtype"]
response_df_proc = response_df_proc.apply(reclassify_impact_subtype, axis=1)
response_df_proc = response_df_proc[response_df_proc["impactSubtype"] != "Unknown"]


In [28]:
response_df_proc["impactSubtype"].value_counts()

impactSubtype
Affected People                                    44
Other Human Impacts                                40
Residential Buildings                              37
Other Infrastructural Impacts                      26
Crop Production and Forestry                       23
Human Health and Wellbeing                         22
Access to Food                                     20
Access to Water, Sanitation, and Hygiene           20
Displaced People                                   19
Access to Healthcare                               18
Homeless People                                    16
Affected Livestock and Animals                     15
Water Quality and Availability                     15
Human Deaths                                       14
Agricultural Infrastructure                        13
Other Economic Activity & Livelihood Production    13
Road Infrastructure                                10
Other Service Access Impacts                        9
Water, Sanitat

In [29]:
#reclassify hazard
response_df_proc["hazards_orig"] = response_df_proc["hazards"]
response_df_proc = response_df_proc.apply(reclassify_hazard, hazard_kw_reclass=hazard_kw_reclass, axis=1)
response_df_proc.hazards.value_counts()

hazards
[Flood]                                                                                                                122
[Flood, Convective storm]                                                                                               46
[Drought]                                                                                                               41
[Flood, Mass movement]                                                                                                  26
[Flood, Tropical storm]                                                                                                 25
[Flood, Epidemic, Conflict]                                                                                             20
[Flood, Epidemic]                                                                                                       18
[Drought, Tropical storm]                                                                                               17
[Tropica

In [30]:
response_df_proc[response_df_proc["appealCode"] == "MDRSD034"][["hazards","hazards_orig", "flag_hazards_reclass"]]

,hazards,hazards_orig,flag_hazards_reclass
27,"[Flood, Epidemic]","[Flood, Epidemic]",False
28,"[Flood, Epidemic]","[Flood, Epidemic]",False
29,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
30,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
31,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
32,"[Flood, Epidemic]","[Flood, Epidemic]",False
33,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
34,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
35,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
36,[Flood],[Flood],False


In [31]:
response_df_proc

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,flag_impactSubtype_reclass,hazards_orig,flag_hazards_reclass
0,Affected People,326788.0,NaN,NaN,exact,people,"[People Affected: 326,788 people]",0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
1,Injured People,584.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
2,Human Deaths,306.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
3,Displaced People,9500.0,NaN,NaN,exact,residents,"[Sindh experienced acute urban flooding, parti...",0,[Pakistan],"[Badin, Dadu, Jacobabad]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
4,Homeless People,15000.0,NaN,NaN,exact,houses,"[In response to this situation, the government...",0,[Pakistan],"[Sindh, Balochistan, Khyber Pakhtunkhwa]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,Other Human Impacts,1.0,NaN,NaN,exact,NaN,[The original plan was to hire an appeal coord...,1,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
484,Other Human Impacts,130.0,NaN,NaN,exact,training sessions,[From 130 to 50 training sessions in hygiene p...,0,[Guatemala],"[El Quiché, Western Temperate Highlands, Guate...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
485,Other Human Impacts,13.0,NaN,NaN,exact,%,"[Appeal Coverage to date: 13 % (269,543 CHF)]",0,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
486,Crop Production and Forestry,NaN,NaN,NaN,approx,NaN,[According to the Food Security Outlook Update...,2,[Guatemala],"[Western Temperate Highlands, low-lying areas ...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False


In [32]:
wrong_haz = response_df_proc.explode("hazards", ignore_index=True).copy()
wrong_haz = wrong_haz[wrong_haz["hazards"] == "Unknown"]
wrong_haz["hazards"]

Series([], Name: hazards, dtype: object)

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent) and special units (money, deaths)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people
3. ~~Handle currencies~~
4. When unknown unit, try to infer it using kw for other impact subtype and reclass to other subtype if match
5. Handle damage vs destroyed houses
6. Parse correctly impact values min and maxs

In [33]:
#response_df_proc["impactValue"] = response_df_proc["impactValueOrig"]
#response_df_proc["impactUnit"] = response_df_proc["impactUnitOrig"]

In [34]:
def infer_unit_from_annotation(x):
    annotation = x["annotation"] if x["annotation"] else x["valueAnnotation"]
    value = x["impactValue"]
    unit = x["impactUnit"]
    #if there is no value we return what's orignally there
    if pd.isnull(value):
        return pd.Series({"impactValue": value, "impactUnit": unit})

    #format numbers in annotation
    annotation = replace_commas_in_numbers(annotation)
    annotation = replace_count_suffixes(annotation)
    annotation = replace_numbers(annotation)

    #format value
    value = format_number(value)

    #find value in annotation
    kw_value = re.search(r"\{value\}", annotation, re.IGNORECASE)
    unit_kw = [kw for kw in unit_kw_reclass.keys() if re.search(unit_kw_reclass[kw], annotation, re.IGNORECASE)]
    if len(unit_kw) == 1:
        return pd.Series({"impactValue": value, "impactUnit": unit_kw[0]})
    else:
        return "Unknown"


In [35]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
## Units reclassification
#replace numbers in units
response_df_proc = response_df_proc.apply(replace_numbers_unit, axis=1)
#convert money
response_df_proc = response_df_proc.apply(convert_monetary_units, axis=1)
#standardize metric units
response_df_proc = response_df_proc.apply(standardize_metric_units, axis=1)
#assign unit type (e.g. surface, volume, mass)
response_df_proc = response_df_proc.apply(assign_unit_type, axis=1)
#harmonize non metric units
response_df_proc = response_df_proc.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
response_df_proc = response_df_proc.apply(convert_unit, axis=1)
#reclassify units
response_df_proc = response_df_proc.apply(reclassify_units, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
response_df_proc = response_df_proc.apply(normalize_people_unit, axis=1)


/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:539: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  elif len(pd.unique(units_parsed)) == 1:
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/sr

Reclassified subtype from Homeless People to Residential Buildings with unit reclass homes and orig unit homes
Reclassified subtype from Agricultural Infrastructure to Affected Livestock and Animals with unit reclass affected animals and orig unit livestock
Reclassified subtype from Road Infrastructure to Displaced People with unit reclass displaced and orig unit people displaced
Reclassified subtype from Water Quality and Availability to Infected and Ill People with unit reclass cases and orig unit cases
Reclassified subtype from Other Infrastructural Impacts to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit latrines
Reclassified subtype from Water Quality and Availability to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit water treatment plants
Reclassified subtype from Other Infrastructural Impacts to Road Infrastructure with unit reclass roads and orig unit roads
Reclassified subtype from O

In [19]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
test_df = pd.DataFrame({"impactUnit": "nomads families", "unit_type": "%", "impactValue": 100, "impactValueMin": np.nan, "impactValueMax":200}, index=[0])
#harmonize non metric units
test_df = test_df.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
test_df = test_df.apply(convert_unit, unit_converter=unit_converter, axis=1)
#reclassify units
test_df = test_df.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
test_df = test_df.apply(normalize_people_unit, axis=1)
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [20]:
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [22]:
## Post conversion flags
country_pop = pd.read_csv(DATA_PATH / ("API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
# country_pop = pd.read_csv(os.path.join(DATA_PATH, "API_SP.POP.TOTL_DS2_en_csv_v2_131993", "API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
response_df_proc["flag_pop_cntry"] = response_df_proc.apply(pop_cntry_check, country_pop=country_pop, axis=1)
response_df_proc["flag_value_no_unit"] = response_df_proc.apply(flag_value_no_unit, axis=1)
response_df_proc["flag_partial_unit"] = response_df_proc.apply(flag_partial_unit, axis=1)
response_df_proc["flag_percent"] = response_df_proc.apply(flag_percent, axis=1)

 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023


In [23]:
# save
response_df_proc.to_csv(DATA_OUT_PROC / savename, index=False)

## Post post process

In [12]:
#load data (model)

res_savename = "labelled_reports_impacts_all_v111025"
#"labelled_reports_DREF_target_meta-llama_llama-4-scout-17b-16e-instruct_v211025"
#"labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv"
#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"
#response_df = pd.read_csv(DATA_OUT_LLMS /(res_savename + ".csv"))
suffix = "_geo_v161025"
res_savename_geo =  "post_processed_new_unit_std_" + res_savename + suffix
response_df_geo = gpd.read_file(DATA_OUT_PROC / (res_savename_geo+".gpkg"))
#load data (labelled)
#res_savename = "labelled_reports_impacts_all_v080925.csv"
#response_df = pd.read_csv(DATA_LABELLED / res_savename)
savename = "merged_subtypes_" + res_savename_geo

In [13]:
#merge infra and service access
IMPACT_SUBTYPE_MERGER = {
    "Transport Infrastructure and Access to Mobility" : r"(Road Infrastructure|Other Transport Infrastructure|Mobility and Access to Transport)",
    "WASH Infrastructure and Access to WASH" : r"(Water, Sanitation, and Hygiene Infrastructure|Access to Water, Sanitation, and Hygiene|Water Quality and Availability)",
    "Healthcare Infrastructure and Access to Healthcare" : r"(Healthcare Infrastructure|Access to Healthcare)",
    "IT and Communication Infrastructure and Access to IT and Communication" : r"(IT Infrastructure and Communication Infrastructure|Access to IT and Communication Infrastructure)",
    "Education Infrastructure and Access to Education" : r"(Education Infrastructure|Access to Education)",
    "Agricultural Infrastructure and Access to Food" : r"(Agricultural Infrastructure|Access to Food|Crop Production and Forestry|Affected Livestock and Animals|Other Agricultural Impacts)",
    "Power and Energy Infrastructure and Access to Power and Energy" : r"(Power and Energy Infrastructure|Access to Power and Energy)",
}
def merge_impact_subtypes(x, impact_kw_reclass=IMPACT_SUBTYPE_MERGER):
    candidates = []
    for key, value in impact_kw_reclass.items():
        if re.search(value, x["impactSubtype"], re.IGNORECASE):
            candidates.append(key)
    if len(candidates) == 1:
        x["impactSubtype"] = candidates[0]
        x["flag_impactSubtype_merged"] = True
    else:
        x["flag_impactSubtype_merged"] = False
    return x
response_df_geo = response_df_geo.apply(merge_impact_subtypes, impact_kw_reclass=IMPACT_SUBTYPE_MERGER,axis=1)


In [14]:
from src.geocoding import atomic_gpkg_save
atomic_gpkg_save(response_df_geo, DATA_OUT_PROC / (savename + ".gpkg"))

True

In [15]:
response_df_geo

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,...,flag_partial_unit,flag_percent,locationLowestAdmin,geocoding_country_flag,geocoding_osm_flag,locationOsm,locationPolygon,iso3_code,geometry,flag_impactSubtype_merged
0,2019-03-14,Affected People,1381000.0,people,exact,NaN,NaN,['SITUATION ANALYSIS Description of the disast...,['China'],"['Deyang', 'Mianyang', 'Guangyuan']",...,False,False,ADM_1,0,0,"['Deyang', 'Mianyang', 'Guangyuan']","['Sichuan Province', 'Sichuan Province', 'Sich...",['CHN'],"MULTIPOLYGON (((104.45871 28.11482, 103.02238 ...",False
1,2019-03-14,Human Deaths,3.0,people,exact,NaN,NaN,['SITUATION ANALYSIS Description of the disast...,['China'],"['Deyang', 'Mianyang', 'Guangyuan']",...,False,False,ADM_1,0,0,"['Deyang', 'Mianyang', 'Guangyuan']","['Sichuan Province', 'Sichuan Province', 'Sich...",['CHN'],"MULTIPOLYGON (((104.45871 28.11482, 103.02238 ...",False
2,2019-03-14,Displaced People,222000.0,people,exact,NaN,NaN,['SITUATION ANALYSIS Description of the disast...,['China'],"['Deyang', 'Mianyang', 'Guangyuan']",...,False,False,ADM_1,0,0,"['Deyang', 'Mianyang', 'Guangyuan']","['Sichuan Province', 'Sichuan Province', 'Sich...",['CHN'],"MULTIPOLYGON (((104.45871 28.11482, 103.02238 ...",False
3,2019-03-14,Injured People,22000.0,people,exact,NaN,NaN,['SITUATION ANALYSIS Description of the disast...,['China'],"['Deyang', 'Mianyang', 'Guangyuan']",...,False,False,ADM_1,0,0,"['Deyang', 'Mianyang', 'Guangyuan']","['Sichuan Province', 'Sichuan Province', 'Sich...",['CHN'],"MULTIPOLYGON (((104.45871 28.11482, 103.02238 ...",False
4,2019-03-14,Residential Buildings,900.0,homes,approx,900.0,NaN,['SITUATION ANALYSIS Description of the disast...,['China'],"['Deyang', 'Mianyang', 'Guangyuan']",...,False,False,ADM_1,0,0,"['Deyang', 'Mianyang', 'Guangyuan']","['Sichuan Province', 'Sichuan Province', 'Sich...",['CHN'],"MULTIPOLYGON (((104.45871 28.11482, 103.02238 ...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,2019-04-24,Residential Buildings,336.0,homes,exact,NaN,NaN,['The initial assessment conducted between 28 ...,['Guinea-Bissau'],"['Bissau (Autonomous Sector of Bissau)', 'Antu...",...,False,False,ADM_0,3,0,['Guinea-Bissau'],"['Guinea-Bissau', 'Guinea-Bissau', 'Guinea-Bis...",['GNB'],"MULTIPOLYGON (((-16.09648 11.19955, -16.09218 ...",False
552,2019-04-24,Education Infrastructure and Access to Education,37.0,education structures,exact,NaN,NaN,['SITUATION ANALYSIS Description of the disast...,['Guinea-Bissau'],['Bissau (Autonomous Sector of Bissau)'],...,False,False,ADM_0,1,0,['Guinea-Bissau'],['Guinea-Bissau'],['GNB'],"MULTIPOLYGON (((-16.09849 11.19945, -16.09648 ...",True
553,2016-11-29,Residential Buildings,750.0,homes,approx,750.0,NaN,"['On 21 June 2016, heavy rain and hailstorm hi...",['Hungary'],['Szabolcs-Szatmr-Bereg County'],...,False,False,ADM_0,1,0,['Hungary'],['Hungary'],['HUN'],"MULTIPOLYGON (((16.11381 46.86909, 16.11381 46...",False
554,2016-11-14,Agricultural Infrastructure and Access to Food,NaN,null,None,NaN,NaN,['The 2015 drought caused families to lose 50 ...,['Guatemala'],"['Chich municipality', 'Patzit municipality']",...,False,False,ADM_0,2,0,['Guatemala'],"['Guatemala', 'Guatemala']",['GTM'],"MULTIPOLYGON (((-90.83696 16.79568, -90.83723 ...",True
